# DE2 — Cahier de Projet Final
Auteur : Samba DIALLO - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

Ceci est l'artefact exécutable principal. Configurez les chemins, lancez le pipeline complet (Batch ETL → Streaming → Traitement de Texte → Itératif → Préparation LLM), et enregistrez les preuves.

### Étape 0 : Configuration & Initialisation de Spark
Avant de commencer tout traitement, nous chargeons nos paramètres centraux depuis `de2_project_config.yml`. Cela nous permet de garder les chemins, les SLOs (objectifs de niveau de service) et les configurations en dehors du code. Nous initialisons également la `SparkSession` en activant l'exécution adaptative des requêtes (AQE).

In [1]:
# ==================================================================
# 0. Charger la configuration et initialiser Spark
# ==================================================================
import yaml, pathlib, datetime, time, json
import os
from pyspark.sql import SparkSession, functions as F, types as T

with open("de2_project_config.yml", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

spark = SparkSession.builder \
    .appName(CFG["spark"]["app_name"]) \
    .master(CFG["spark"]["master"]) \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print("Spark:", spark.version)
print("UI:", spark.sparkContext.uiWebUrl)
print("Configuration chargée avec succès.")

Spark: 4.1.1
UI: http://samba:4040
Configuration chargée avec succès.


## Phase 1: Pipeline Batch ETL & Architecture Medallion

Cette phase ingère les données brutes de GitHub Archive, les traite à travers les couches Bronze, Silver et Gold, et enregistre les métriques de performance du pipeline.

### 1.1 Couche Bronze (Ingestion Brute)
La couche Bronze est responsable de l'ingestion des données brutes sans altérer la structure de base. Nous lisons les fichiers `.json.gz` depuis notre dossier `archive/`, nous ajoutons une colonne `ingestion_tstamp` pour l'audit, et nous sauvegardons les données efficacement au format **Parquet**. Nous enregistrons également le temps d'exécution pour vérifier nos SLOs.

In [2]:
# 1.1 Couche Bronze (Ingestion Brute)
def process_bronze():
    print("Démarrage du traitement de la couche Bronze...")
    t0 = time.time()
    # Lire les données JSON brutes
    raw_df = spark.read.json(CFG["paths"]["raw_csv_glob"])
    
    # Ajouter les métadonnées d'ingestion
    bronze_df = raw_df.withColumn("ingestion_tstamp", F.current_timestamp())
    
    # Sauvegarder au format Parquet
    bronze_df.write.mode("overwrite").parquet(CFG["paths"]["bronze"])
    
    t1 = time.time()
    record_metric("Batch ETL", "bronze_latency_sec", t1 - t0, f"Traitement de {bronze_df.count()} évènements bruts")
    print(f"Couche Bronze terminée en {t1 - t0:.2f} secondes.")
    return bronze_df

def record_metric(stage, metric_name, metric_value, notes=""):
    timestamp = datetime.datetime.now().isoformat()
    run_id = f"run_{int(time.time())}"
    row = f"{run_id},{stage},task,{metric_name},{metric_value},{notes},{timestamp}\n"
    with open(CFG["paths"]["metrics_log"], "a", encoding="utf-8") as f:
        f.write(row)


### 1.2 Couche Silver (Nettoyage & Application du Schéma)
La couche Silver applique des validations strictes sur le schéma et aplatit les données. Nous filtrons les enregistrements malformés (ex: `id` manquant), extrayons les éléments nécessaires depuis les JSON imbriqués (comme `actor.login`, `repo.name`), convertissons les chaînes de caractères en types `timestamp`/`date`, et écrivons le résultat partitionné par `date` pour optimiser les requêtes futures.

In [3]:
# 1.2 Couche Silver (Nettoyage et Application du Schéma)
def process_silver():
    print("Démarrage du traitement de la couche Silver...")
    t0 = time.time()
    bronze_df = spark.read.parquet(CFG["paths"]["bronze"])
    
    # Filtrer les identifiants vides et sélectionner les colonnes importantes
    # Extraction des structures JSON imbriquées
    silver_df = bronze_df.filter(F.col("id").isNotNull()) \
        .select(
            F.col("id").alias("event_id"),
            F.col("type").alias("event_type"),
            F.col("actor.login").alias("actor_login"),
            F.col("repo.name").alias("repo_name"),
            F.col("created_at").cast("timestamp").alias("created_at"),
            F.to_date(F.col("created_at")).alias("date")
        )
    
    # Partitionner par date et sauvegarder
    silver_df.write.mode("overwrite") \
        .partitionBy(*CFG["layout"]["partition_by"]) \
        .parquet(CFG["paths"]["silver"])
    
    t1 = time.time()
    record_metric("Batch ETL", "silver_latency_sec", t1 - t0, f"Nettoyage de {silver_df.count()} évènements")
    print(f"Couche Silver terminée en {t1 - t0:.2f} secondes.")
    return silver_df


### 1.3 Couche Gold (Agrégations Analytiques)
La couche Gold crée des datasets agrégés prêts à être consommés par des tableaux de bord analytiques ou des outils de BI. Ici, nous calculons le volume quotidien de chaque type d'événement par dépôt de code (`repo_activity`). Cela réduit considérablement la taille des données et optimise la vitesse des rapports.

In [4]:
# 1.3 Couche Gold (Agrégations)
def process_gold():
    print("Démarrage du traitement de la couche Gold...")
    t0 = time.time()
    silver_df = spark.read.parquet(CFG["paths"]["silver"])
    
    # Agréger les événements par dépôt et par jour
    gold_repo_activity = silver_df.groupBy("date", "repo_name", "event_type") \
        .agg(F.count("event_id").alias("event_count"))
        
    gold_repo_activity.write.mode("overwrite").parquet(os.path.join(CFG["paths"]["gold"], "repo_activity"))
    
    t1 = time.time()
    record_metric("Batch ETL", "gold_latency_sec", t1 - t0, f"Agrégation des dépôts réussie")
    print(f"Couche Gold terminée en {t1 - t0:.2f} secondes.")
    return gold_repo_activity


### Exécution du Pipeline Batch (Phase 1)
Il est temps de déclencher le traitement réel des données. Décommentez les appels de fonctions ci-dessous pour lancer le pipeline de bout en bout sur vos données GitHub Archive locales.

In [ ]:
# Exécuter le Pipeline de la Phase 1
# À décommenter et exécuter après le téléchargement via download_gh_archive.py
process_bronze()
process_silver()
process_gold()


Démarrage du traitement de la couche Bronze...
